In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier
Configuration: {'run_name': 'experiment_with_10_classes', 'seed': 42, 'n_classes': 10, 'cutoff_year': 1996, 'data_exploration_dir': 'experiment_with_10_classes/data_exploration', 'artifacts_dir': 'experiment_with_10_classes/artifacts', 'embeddings_dir': 'experiment_with_10_classes/embeddings', 'models_dir': 'experiment_with_10_classes/models', 'results_dir': 'experiment_with_10_classes/results', 'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False, 'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3', 'weight_decay': '1e-4', 'early_stopping': 3, 'test_split': 0.2, 'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}
DATA_EXPLORATION_DIR: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/data_exploration
ARTIFACTS_DIR: /home/marcmaceira/pro

# Reuters News Topic Classification - Model Training

This notebook implements and trains different models for the Reuters news topic classification task. We'll compare several approaches:

1. Multinomial Naive Bayes with TF-IDF
2. Linear SVM with TF-IDF (unigrams)
3. Linear SVM with TF-IDF (bigrams)
4. MiniLM embeddings with Logistic Regression
5. RAG-kMajority
6. RAG-CentroidNN
7. RAG-LLM

## Setup and Data Loading

In [2]:
import logging
import warnings
import random
import os
import numpy as np
from src.datasets.dataset import load_data
from src.algorithms.naive_bayes import NaiveBayesClassifier
from src.algorithms.linear_svm import LinearSVMClassifier, LinearSVMBigrams
from src.algorithms.transformer_logreg import TransformerLogReg
from src.rag import load_kmajority, load_centroid, load_llm
from src.rag.adapter_sklearn import RagSklearnAdapter
from src.evaluation import run_evaluations
from src.embeddings.openai_embedder import OpenAIEmbedder
from src.rag.vector_store import VectorStore



warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

# Load dataset
X_train, y_train, X_test, y_test, label_names = load_data(N_CLASSES)
print(f'Train docs: {len(X_train):,},  Test docs: {len(X_test):,}')
print('Labels:', label_names)

/home/marcmaceira/venv/reuters-rag-classifier/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-04 04:32:53,381 - faiss.loader - INFO - Loading faiss with AVX2 support.
2025-05-04 04:32:53,406 - faiss.loader - INFO - Successfully loaded faiss with AVX2 support.
2025-05-04 04:32:53,413 - faiss - INFO - Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.


Train docs: 6,337,  Test docs: 2,477
Labels: ['earn', 'acq', 'crude', 'interest', 'money-fx', 'trade', 'grain', 'corn', 'dlr', 'money-supply']


## Initialize and Train All Models

In [3]:
# ensure we’re using OpenAI for embeddings
os.environ['USE_OPENAI_EMBEDDINGS'] = '1'
# OR explicitly pass use_openai=True below

# instantiate the OpenAI embedder (batch size adjustable)
openai_embedder = OpenAIEmbedder(model="text-embedding-3-small", batch_size=50)


In [4]:
# Initialize all models
models = {
    'Naive Bayes':       NaiveBayesClassifier(),
    'Linear SVM':        LinearSVMClassifier(),
    'TF-IDF bigrams + SVM': LinearSVMBigrams(),
    'MiniLM + LogReg':   TransformerLogReg(),
    # RAG variants all take an embedder under the hood:
    'RAG-kMajority':     RagSklearnAdapter(load_kmajority(top_k=5)),
    'RAG-CentroidNN':    RagSklearnAdapter(load_centroid()),
    # For the LLM‐backed RAG we also pass the same embedder plus your LLM choice:
    'RAG-LLM (OpenAI-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            embedder=openai_embedder,
            use_openai=True  # Add this parameter to use the OpenAI index
        )
    ),
    # Add the local embeddings variant:
    'RAG-LLM (local-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            use_openai=False,  # Explicitly specify to use local index
            embedder=lambda texts: VectorStore.embed("sentence-transformers/all-MiniLM-L6-v2", texts)
        )
    ),
}


if 0:
    # Initialize all models
    models = {

        # For the LLM‐backed RAG we also pass the same embedder plus your LLM choice:
        'RAG-LLM (OpenAI-embeddings)': RagSklearnAdapter(
            load_llm(
                top_k=5,
                model="gpt-4o-mini",
                embedder=openai_embedder,
                use_openai=True  # Add this parameter to use the OpenAI index
            )
        ),
    }

2025-05-04 04:32:55,637 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2025-05-04 04:32:55,638 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2025-05-04 04:32:57,754 - src.rag.retrieval - INFO - Loading SentenceTransformer retriever
2025-05-04 04:32:57,756 - src.rag.retrieval - INFO - Using index: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss, meta: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/meta.jsonl
2025-05-04 04:32:57,757 - src.rag.vector_store - INFO - Loading FAISS index from /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss
2025-05-04 04:32:57,774 - src.rag.vector_store - INFO - Loading metadata from /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/meta.jsonl
2025-05-04 04:32:59,17

In [5]:
from src.training import run_trainings
from src.evaluation import run_evaluations
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


# ▸  train + persist
trained = run_trainings(models, X_train, y_train,
                        output_dir=MODELS_DIR)




Training: 100%|██████████| 1/1 [00:00<00:00, 668.73it/s]

Saved model to /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/models/RAG-LLM (OpenAI-embeddings).joblib


In [7]:
import pandas as pd
# Dictionary to collect all evaluation results
all_results = {}

# Evaluate each model and collect results with enhanced logging
for name, model in trained.items():

    output_dir = os.path.join(RESULTS_DIR, name)

    # Check if evaluation files already exist
    test_report_path = os.path.join(output_dir, "test_report.csv")
    test_confusion_path = os.path.join(output_dir, "test_confusion.png")
    
    if os.path.exists(test_report_path) and os.path.exists(test_confusion_path):
        
        logging.info(f"Evaluation files already exist for {name}, loading results from {output_dir}")
        # Load the saved report to recreate the results dictionary
        report_df = pd.read_csv(test_report_path, index_col=0)
        report_dict = report_df.to_dict('index')
        
        # Reconstruct results dictionary
        result = {
            'test': {
                'accuracy': report_dict['accuracy']['f1-score'] if 'accuracy' in report_dict else report_dict['accuracy.1']['f1-score'],
                'macro_f1': report_dict['macro avg']['f1-score'],
                'weighted_f1': report_dict['weighted avg']['f1-score'],
                'report': report_dict
            }
        }
        
        # Also check for train results
        train_report_path = os.path.join(output_dir, "train_report.csv")
        if os.path.exists(train_report_path):
            train_df = pd.read_csv(train_report_path, index_col=0)
            train_dict = train_df.to_dict('index')
            result['train'] = {
                'accuracy': train_dict['accuracy']['f1-score'] if 'accuracy' in train_dict else train_dict['accuracy.1']['f1-score'],
                'macro_f1': train_dict['macro avg']['f1-score'],
                'weighted_f1': train_dict['weighted avg']['f1-score'],
                'report': train_dict
            }

    result = run_evaluations(
        model,                 # or f"artifacts/models/{name}.joblib"
        X_test, y_test,
        X_train=X_train, y_train=y_train,
        output_dir=output_dir,
        model_name=name  # Pass the model name for better logging
    )
    all_results[name] = result



    

2025-05-04 04:33:11,385 - root - INFO - 🔍 Evaluating model: RAG-LLM (OpenAI-embeddings)
2025-05-04 04:33:11,387 - root - INFO -   • Evaluating on test set (2477 samples)...
2025-05-04 04:33:11,391 - src.rag.rag_llm - INFO - Starting prediction for 2477 documents with batch size 8
2025-05-04 04:33:11,392 - src.embeddings.openai_embedder - INFO - Starting OpenAI embedding generation for 2477 texts with model text-embedding-3-small
2025-05-04 04:33:11,393 - src.embeddings.openai_embedder - INFO - Processing embedding batch 1 with 50 texts
2025-05-04 04:33:12,466 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-04 04:33:13,053 - src.embeddings.openai_embedder - INFO - Batch 1 used 13486 tokens
2025-05-04 04:33:13,054 - src.embeddings.openai_embedder - INFO - Batch 1 completed in 1.66 seconds
2025-05-04 04:33:13,056 - src.embeddings.openai_embedder - INFO - Average time per text in batch: 0.0332 seconds
2025-05-04 04:33:13,056 - src.embeddin

In [ ]:

# After all evaluations, print a comparison summary
logging.info("📊 Model Performance Comparison:")
for name, result in all_results.items():
    test_acc = result['test']['accuracy']
    test_f1 = result['test']['macro_f1']
    
    if 'train' in result:
        train_acc = result['train']['accuracy']
        train_f1 = result['train']['macro_f1']
        logging.info(f"{name:20} | Test acc: {test_acc:.4f}, f1: {test_f1:.4f} | Train acc: {train_acc:.4f}, f1: {train_f1:.4f}")
    else:
        logging.info(f"{name:20} | Test acc: {test_acc:.4f}, f1: {test_f1:.4f} | Train: N/A")

## Model Comparison

In [11]:

# Organize results using our new function
from src.evaluation import collect_evaluation_results, compare_models, print_evaluation_results
overall_results, class_results = collect_evaluation_results(all_results)

# Create a DataFrame for comparison
results_df = compare_models(overall_results)
print(results_df)

# Use our utility functions to organize and print results
print_evaluation_results(overall_results, class_results)

             Accuracy  Macro F1  Weighted F1
Naive Bayes     0.928     0.822        0.928
Linear SVM      0.947     0.867        0.947
Overall Model Comparison:
             accuracy  macro_f1  weighted_f1  macro_precision  macro_recall  \
Naive Bayes     0.928     0.822        0.928            0.854         0.808   
Linear SVM      0.947     0.867        0.947            0.874         0.862   

             weighted_precision  weighted_recall  
Naive Bayes               0.931            0.928  
Linear SVM                0.947            0.947  

Per-Class Model Comparison:

Class: acq
             precision  recall  f1-score  support
Naive Bayes      0.958   0.975     0.966    719.0
Linear SVM       0.974   0.976     0.975    719.0

Class: corn
             precision  recall  f1-score  support
Naive Bayes      0.868   0.688     0.767     48.0
Linear SVM       0.949   0.771     0.851     48.0

Class: crude
             precision  recall  f1-score  support
Naive Bayes      0.916   0.962

## Save model and metrics

In [9]:
# Save all models
import joblib
import os
from src.utils.model_storage import save_model

# Save all models
for name, model in models.items():
    save_model(
        model=model,
        model_name=name,
        metrics=all_results[name],
        n_classes=N_CLASSES
    )

NameError: name 'results' is not defined

## Next Steps

In the next notebook, we'll:
1. Perform detailed error analysis
2. Create a confusion matrix to understand model mistakes
3. Implement a semantic search demo using the transformer embeddings